### Завдання 1. Читання, вибір колонок і фільтрація

In [0]:


# 1. Читання таблиці з каталогу samples
trips_raw = spark.read.table("samples.nyctaxi.trips")

# 2. Вибір необхідних колонок, видалення null у поштових індексах та фільтрація значень
trips_clean = (
    trips_raw
    # Залишаємо лише 5 необхідних колонок
    .select(
        "tpep_pickup_datetime", 
        "trip_distance", 
        "fare_amount", 
        "pickup_zip", 
        "dropoff_zip"
    )
    # Вилучаємо null лише у колонках поштових індексів (підказка про окремий параметр)
    .dropna(subset=["pickup_zip", "dropoff_zip"])
    # Фільтруємо відстань та вартість (мають бути більші за нуль)
    .filter(trips_raw.trip_distance > 0)
    .filter(trips_raw.fare_amount > 0)
)

# 3. Виведення перших 5 рядків для перевірки результату
display(trips_clean.limit(5))




tpep_pickup_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2.6,18.5,10199,10022


### Завдання 2. Нові колонки

In [0]:


import pyspark.sql.functions as F

# Создаем trips_prepared на основе trips_clean, добавляя две новые колонки
trips_prepared = (
    trips_clean
    # 1. Номер месяца из даты посадки
    .withColumn("pickup_month", F.month(F.col("tpep_pickup_datetime")))
    # 2. Стоимость одной мили (fare_amount делим на trip_distance)
    .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
)

# Выводим первые 5 строк для проверки (будет 7 колонок)
display(trips_prepared.limit(5))





tpep_pickup_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip,pickup_month,fare_per_mile
2016-02-13T21:47:53.000Z,1.4,8.0,10103,10110,2,5.714285714285714
2016-02-13T18:29:09.000Z,1.31,7.5,10023,10023,2,5.7251908396946565
2016-02-06T19:40:58.000Z,1.8,9.5,10001,10018,2,5.277777777777778
2016-02-12T19:06:43.000Z,2.3,11.5,10044,10111,2,5.0
2016-02-23T10:27:56.000Z,2.6,18.5,10199,10022,2,7.115384615384615


### Завдання 3


In [0]:
%sql
-- Создаем том, если он еще не существует, чтобы код не падал при повторном запуске
CREATE VOLUME IF NOT EXISTS workspace.default.taxi_files;


In [0]:
# 1. Сохраняем путь к папке в переменную (подсказка: путь понадобится трижды)
path_trips = "/Volumes/workspace/default/taxi_files/trips_prepared"

# 2. Запись DataFrame trips_prepared в формате Parquet с партиционированием
(
    trips_prepared.write
    .mode("overwrite")                  # Перезаписываем данные без дублей и ошибок
    .partitionBy("pickup_month")        # Раскладываем по подпапкам pickup_month=значение
    .format("parquet")                  # Задаем формат Parquet
    .save(path_trips)                   # Указываем путь сохранения
)

# 3. Вывод перечня содержимого папки с помощью утилиты dbutils
display(dbutils.fs.ls(path_trips))

# 4. Чтение записанных файлов в новый DataFrame trips_from_files
trips_from_files = spark.read.format("parquet").load(path_trips)

# 5. Вывод первых 5 строк для проверки (колонка pickup_month восстановится из названий папок)
display(trips_from_files.limit(5))


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/taxi_files/trips_prepared/_SUCCESS,_SUCCESS,0,1792435419000
dbfs:/Volumes/workspace/default/taxi_files/trips_prepared/pickup_month=1/,pickup_month=1/,0,1792435419081
dbfs:/Volumes/workspace/default/taxi_files/trips_prepared/pickup_month=2/,pickup_month=2/,0,1792435419081


tpep_pickup_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip,fare_per_mile,pickup_month
2016-02-13T21:47:53.000Z,1.4,8.0,10103,10110,5.714285714285714,2
2016-02-13T18:29:09.000Z,1.31,7.5,10023,10023,5.7251908396946565,2
2016-02-06T19:40:58.000Z,1.8,9.5,10001,10018,5.277777777777778,2
2016-02-12T19:06:43.000Z,2.3,11.5,10044,10111,5.0,2
2016-02-23T10:27:56.000Z,2.6,18.5,10199,10022,7.115384615384615,2


### Завдання 4. Групування й агрегація

In [0]:
import pyspark.sql.functions as F

# 1. Создаем DataFrame pickups_by_zip (агрегация по месту посадки)
pickups_by_zip = (
    trips_from_files
    .groupBy("pickup_zip")
    .agg(
        F.count("*").alias("trips_started"),
        F.sum("fare_amount").alias("total_fare"),
        F.avg("fare_per_mile").alias("avg_fare_per_mile")
    )
)

# 2. Создаем DataFrame dropoffs_by_zip (агрегация по месту высадки)
dropoffs_by_zip = (
    trips_from_files
    .groupBy("dropoff_zip")
    .agg(
        F.count("*").alias("trips_finished")
    )
)

# 3. Выводим результаты, отсортированные по возрастанию почтовых индексов
display(pickups_by_zip.orderBy("pickup_zip"))
display(dropoffs_by_zip.orderBy("dropoff_zip"))


pickup_zip,trips_started,total_fare,avg_fare_per_mile
7002,2,21.0,7.492672203224231
7030,1,40.0,6.34920634920635
7087,1,31.0,3.5755478662053055
7311,1,60.0,30.0
7718,1,5.0,6.25
7737,1,15.5,5.308219178082192
10001,1227,13028.01,6.272036894787021
10002,611,7439.5,5.4275164904637325
10003,1180,12963.0,7.608240110891416
10005,101,1602.5,4.740302442734421


dropoff_zip,trips_finished
7002,2
7008,1
7024,2
7030,4
7087,1
7114,28
7201,1
7302,1
7304,1
7310,1


### Завдання 5. Зʼєднання і значення null

In [0]:
# Создаем zip_joined с помощью левого соединения (left join)
zip_joined = dropoffs_by_zip.join(
    pickups_by_zip,
    on=dropoffs_by_zip.dropoff_zip == pickups_by_zip.pickup_zip,
    how="left"
)

# Выводим результат на экран
display(zip_joined)


dropoff_zip,trips_finished,pickup_zip,trips_started,total_fare,avg_fare_per_mile
10022,482,10022,517,5090.0,6.205664089593452
10002,486,10002,611,7439.5,5.4275164904637325
10103,472,10103,580,7285.51,6.263780619520343
10278,160,10278,167,2169.5,5.686703578072423
10280,91,10280,103,1681.5,5.206186256436832
10465,3,null,null,null,null
7302,1,null,null,null,null
10710,1,null,null,null,null
11575,1,null,null,null,null
10003,929,10003,1180,12963.0,7.608240110891416


In [0]:
# Создаем zips_without_pickups, фильтруя строки, где pickup_zip равен null
zips_without_pickups = zip_joined.filter(F.col("pickup_zip").isNull())

# Выводим результат на экран
display(zips_without_pickups)


dropoff_zip,trips_finished,pickup_zip,trips_started,total_fare,avg_fare_per_mile
10465,3,null,null,null,null
7302,1,null,null,null,null
10710,1,null,null,null,null
11575,1,null,null,null,null
7304,1,null,null,null,null
10462,3,null,null,null,null
11415,2,null,null,null,null
10463,9,null,null,null,null
11360,2,null,null,null,null
11010,1,null,null,null,null


In [0]:
import pyspark.sql.functions as F

# 1. Заменяем null на 0 только в колонках trips_started и total_fare
zip_filled = zip_joined.fillna(0, subset=["trips_started", "total_fare"])

# 2. Формируем итоговый DataFrame zip_report со всеми вычислениями и переименованиями
zip_report = (
    zip_filled
    # Переименовываем dropoff_zip в zip
    .withColumnRenamed("dropoff_zip", "zip")
    # Считаем разницу net_flow (уже после того, как заменили null на 0)
    .withColumn("net_flow", F.col("trips_finished") - F.col("trips_started"))
    # Выбираем и расставляем колонки строго в требуемом порядке
    .select(
        "zip",
        "trips_finished",
        "trips_started",
        "net_flow",
        "total_fare",
        "avg_fare_per_mile"  # Здесь null сохраняется для индексов без посадок
    )
    # Сортируем по zip за возрастанием
    .orderBy("zip")
)

# 3. Выводим итоговый отчет на экран
display(zip_report)


zip,trips_finished,trips_started,net_flow,total_fare,avg_fare_per_mile
7002,2,2,0,21.0,7.492672203224231
7008,1,0,1,0.0,null
7024,2,0,2,0.0,null
7030,4,1,3,40.0,6.34920634920635
7087,1,1,0,31.0,3.5755478662053055
7114,28,0,28,0.0,null
7201,1,0,1,0.0,null
7302,1,0,1,0.0,null
7304,1,0,1,0.0,null
7310,1,0,1,0.0,null


### Завдання 6. Конвеєр одним ланцюжком і запит SQL

In [0]:
import pyspark.sql.functions as F

# Создаем pickups_chain одним непрерывным ланцюжком
pickups_chain = (
    spark.read.table("samples.nyctaxi.trips")
    # Шаг 1: Выбор колонок и фильтрация (Завдання 1)
    .select("tpep_pickup_datetime", "trip_distance", "fare_amount", "pickup_zip", "dropoff_zip")
    .dropna(subset=["pickup_zip", "dropoff_zip"])
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
    # Шаг 2: Расчет стоимости мили (Завдання 2)
    .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
    # Шаг 3: Группировка и агрегация (Завдання 4)
    .groupBy("pickup_zip")
    .agg(
        F.count("*").alias("trips_started"),
        F.sum("fare_amount").alias("total_fare"),
        F.avg("fare_per_mile").alias("avg_fare_per_mile")
    )
    # Шаг 4: Сортировка по возрастанию
    .orderBy("pickup_zip")
)

# Выводим результат
display(pickups_chain)


pickup_zip,trips_started,total_fare,avg_fare_per_mile
7002,2,21.0,7.492672203224231
7030,1,40.0,6.34920634920635
7087,1,31.0,3.5755478662053055
7311,1,60.0,30.0
7718,1,5.0,6.25
7737,1,15.5,5.308219178082192
10001,1227,13028.01,6.27203689478702
10002,611,7439.5,5.427516490463737
10003,1180,12963.0,7.608240110891412
10005,101,1602.5,4.74030244273442


In [0]:
# Создаем pickups_sql с помощью чистого SQL-запроса к исходной таблице
pickups_sql = spark.sql("""
    SELECT 
        pickup_zip,
        COUNT(*) AS trips_started,
        SUM(fare_amount) AS total_fare,
        AVG(fare_amount / trip_distance) AS avg_fare_per_mile
    FROM 
        samples.nyctaxi.trips
    WHERE 
        pickup_zip IS NOT NULL 
        AND dropoff_zip IS NOT NULL
        AND trip_distance > 0
        AND fare_amount > 0
    GROUP BY 
        pickup_zip
    ORDER BY 
        pickup_zip ASC
""")

# Выводим результат в отдельной ячейке
display(pickups_sql)


pickup_zip,trips_started,total_fare,avg_fare_per_mile
7002,2,21.0,7.492672203224231
7030,1,40.0,6.34920634920635
7087,1,31.0,3.5755478662053055
7311,1,60.0,30.0
7718,1,5.0,6.25
7737,1,15.5,5.308219178082192
10001,1227,13028.01,6.27203689478702
10002,611,7439.5,5.427516490463737
10003,1180,12963.0,7.608240110891412
10005,101,1602.5,4.74030244273442


### Завдання 7. Запис у таблицю і прибирання

In [0]:
# 1. Сохраняем DataFrame zip_report в таблицу каталога
(
    zip_report.write
    .mode("overwrite")                           # Перезаписываем содержимое без ошибок
    .format("delta")                             # Используем стандартный формат Delta для таблиц Databricks
    .saveAsTable("workspace.default.zip_report") # Указываем полное имя таблицы
)

# 2. Читаем сохраненную таблицу из каталога
final_report_from_table = spark.read.table("workspace.default.zip_report")

# 3. Выводим её, отсортированную по zip по возрастанию
display(final_report_from_table.orderBy("zip"))


zip,trips_finished,trips_started,net_flow,total_fare,avg_fare_per_mile
7002,2,2,0,21.0,7.492672203224231
7008,1,0,1,0.0,null
7024,2,0,2,0.0,null
7030,4,1,3,40.0,6.34920634920635
7087,1,1,0,31.0,3.5755478662053055
7114,28,0,28,0.0,null
7201,1,0,1,0.0,null
7302,1,0,1,0.0,null
7304,1,0,1,0.0,null
7310,1,0,1,0.0,null


In [0]:
# 1. Сначала удаляем созданную таблицу через SQL
spark.sql("DROP TABLE IF EXISTS workspace.default.zip_report")

# 2. Удаляем папку со всеми Parquet-файлами (использовали переменную пути из Задания 3)
path_trips = "/Volumes/workspace/default/taxi_files/trips_prepared"
dbutils.fs.rm(path_trips, recurse=True)

# 3. Наконец, удаляем сам том (Volume) через SQL
spark.sql("DROP VOLUME IF EXISTS workspace.default.taxi_files")

print("Все созданные объекты успешно удалены из рабочего пространства!")


Все созданные объекты успешно удалены из рабочего пространства!
